# Add Game Number Attribute to FPL Data

This notebook adds a `game_number` attribute (1-38) to:
- `all_seasons_data.csv` - Using fixtures.csv as source of truth
- `defensive_stats_raw.csv` - Using date-based matching

**Approach:** Use fixtures.csv files (actual match data) to create chronological game numbering, then join to player data via fixture ID.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

## Load Datasets

In [57]:
# Load the datasets
print("Loading all_seasons_data.csv...")
all_seasons_df = pd.read_csv('all_seasons_data.csv')

print("Loading defensive_stats_raw.csv...")
defensive_stats_df = pd.read_csv('defensive_stats_raw.csv')

print(f"\n✓ All seasons data shape: {all_seasons_df.shape}")
print(f"✓ Defensive stats shape: {defensive_stats_df.shape}")

print(f"\nSeasons in all_seasons_data: {sorted(all_seasons_df['season'].unique())}")
print(f"Seasons in defensive_stats: {sorted(defensive_stats_df['season'].unique())}")

Loading all_seasons_data.csv...


C:\Users\pc\AppData\Local\Temp\ipykernel_6380\3641664672.py:3: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  all_seasons_df = pd.read_csv('all_seasons_data.csv')


Loading defensive_stats_raw.csv...

✓ All seasons data shape: (216537, 42)
✓ Defensive stats shape: (65789, 29)

Seasons in all_seasons_data: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Seasons in defensive_stats: [np.float64(1920.0), np.float64(2021.0), np.float64(2122.0), np.float64(2223.0), np.float64(2324.0), np.float64(2425.0), np.float64(nan)]


C:\Users\pc\AppData\Local\Temp\ipykernel_6380\3641664672.py:6: DtypeWarning: Columns (10,11,12,13,14,15,16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  defensive_stats_df = pd.read_csv('defensive_stats_raw.csv')


In [58]:
# Display sample data and check key columns
print("Sample from all_seasons_data:")
print(all_seasons_df[['name', 'team', 'season', 'fixture', 'GW', 'kickoff_time', 'was_home']].head(10))

print(f"\nSeasons with fixtures.csv available: 2018-19 to 2024-25")
print(f"Seasons without fixtures.csv: 2016-17, 2017-18")

# Check if game_number column already exists
if 'game_number' in all_seasons_df.columns:
    print("\n⚠️ game_number column already exists - will be replaced")

Sample from all_seasons_data:
                       name     team   season  fixture  GW  \
0              Aaron_Ramsey  Arsenal  2016-17        8   1   
1            Alexis_Sánchez  Arsenal  2016-17        8   1   
2                Alex_Iwobi  Arsenal  2016-17        8   1   
3   Alex_Oxlade-Chamberlain  Arsenal  2016-17        8   1   
4            Carl_Jenkinson  Arsenal  2016-17        8   1   
5               Chuba_Akpom  Arsenal  2016-17        8   1   
6             Danny_Welbeck  Arsenal  2016-17        8   1   
7              David_Ospina  Arsenal  2016-17        8   1   
8          Francis_Coquelin  Arsenal  2016-17        8   1   
9  Gabriel Armando_de Abreu  Arsenal  2016-17        8   1   

                kickoff_time  was_home  
0  2016-08-14 15:00:00+00:00      True  
1  2016-08-14 15:00:00+00:00      True  
2  2016-08-14 15:00:00+00:00      True  
3  2016-08-14 15:00:00+00:00      True  
4  2016-08-14 15:00:00+00:00      True  
5  2016-08-14 15:00:00+00:00      True  


## Step 1: Build Team-Game Mapping from fixtures.csv

For each season (2018-19 to 2024-25):
1. Load fixtures.csv and teams.csv
2. Create team_id → team_name mapping
3. Convert each fixture to 2 team-game records (home + away)
4. Sort chronologically and assign game_number 1-38

In [66]:
def build_game_number_mapping():
    """
    Build a comprehensive mapping of (season, fixture_id) → game_number
    using fixtures.csv as the source of truth.
    
    Only processes seasons that have BOTH fixtures.csv AND teams.csv to ensure
    accurate team ID to name mapping.
    
    Returns:
        - fixture_to_game_number: DataFrame with (season, fixture_id, team_name, game_number)
    """
    
    # Season folders to process (only those with teams.csv)
    season_folders = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
    
    all_mappings = []
    
    for season in season_folders:
        fixtures_path = f'data/{season}/fixtures.csv'
        teams_path = f'data/{season}/teams.csv'
        
        try:
            # Load fixtures and teams
            fixtures_df = pd.read_csv(fixtures_path)
            teams_df = pd.read_csv(teams_path)
            
            # Create team_id → team_name mapping
            team_id_to_name = dict(zip(teams_df['id'], teams_df['name']))
            
            # Convert kickoff_time to datetime
            fixtures_df['kickoff_time'] = pd.to_datetime(fixtures_df['kickoff_time'])
            
            # Keep only finished matches
            finished_fixtures = fixtures_df[fixtures_df['finished'] == True].copy()
            
            if len(finished_fixtures) == 0:
                print(f"{season}: No finished matches found, skipping")
                continue
            
            # Create team-game records (2 per fixture: home + away)
            team_games = []
            
            for _, fixture in finished_fixtures.iterrows():
                fixture_id = fixture['id']
                kickoff_time = fixture['kickoff_time']
                gw = fixture['event']
                
                # Home team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_h'],
                    'team_name': team_id_to_name.get(fixture['team_h'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': True
                })
                
                # Away team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_a'],
                    'team_name': team_id_to_name.get(fixture['team_a'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': False
                })
            
            season_df = pd.DataFrame(team_games)
            
            # Sort by team and kickoff_time (chronological order)
            season_df = season_df.sort_values(['team_name', 'kickoff_time'])
            
            # Assign game_number per team (1, 2, 3, ..., up to 38)
            season_df['game_number'] = season_df.groupby('team_name').cumcount() + 1
            
            all_mappings.append(season_df)
            
            print(f"{season}: {len(finished_fixtures)} fixtures → {len(season_df)} team-game records")
            print(f"         Teams: {season_df['team_name'].nunique()}, Max game_number: {season_df['game_number'].max()}")
            
        except FileNotFoundError as e:
            print(f"{season}: Required file not found, skipping ({e})")
        except Exception as e:
            print(f"{season}: Error - {e}")
    
    # Combine all seasons
    if all_mappings:
        full_mapping = pd.concat(all_mappings, ignore_index=True)
        print(f"\n✅ Total mapping records: {len(full_mapping):,}")
        print(f"Note: Seasons 2016-17, 2017-18, 2018-19 excluded (missing teams.csv)")
        return full_mapping
    else:
        print("❌ No mappings created!")
        return None

# Build the mapping
print("="*80)
print("STEP 1: Building game_number mapping from fixtures.csv")
print("="*80)
game_number_mapping = build_game_number_mapping()

STEP 1: Building game_number mapping from fixtures.csv
2019-20: 380 fixtures → 760 team-game records
         Teams: 20, Max game_number: 38
2020-21: 380 fixtures → 760 team-game records
         Teams: 20, Max game_number: 38
2021-22: 380 fixtures → 760 team-game records
         Teams: 20, Max game_number: 38
2022-23: 380 fixtures → 760 team-game records
         Teams: 20, Max game_number: 38
2023-24: 380 fixtures → 760 team-game records
         Teams: 20, Max game_number: 38
2024-25: 380 fixtures → 760 team-game records
         Teams: 20, Max game_number: 38
2025-26: 90 fixtures → 180 team-game records
         Teams: 20, Max game_number: 9

✅ Total mapping records: 4,740
Note: Seasons 2016-17, 2017-18, 2018-19 excluded (missing teams.csv)


In [67]:
# Verify the mapping looks correct
print("="*80)
print("VERIFICATION: Sample of game_number mapping")
print("="*80)

# Show sample for one team in one season
sample_team = game_number_mapping[
    (game_number_mapping['season'] == '2019-20') & 
    (game_number_mapping['team_name'] == 'Liverpool')
][['team_name', 'game_number', 'gw', 'kickoff_time', 'fixture_id']].head(10)

print("\nLiverpool 2019-20 first 10 games:")
print(sample_team)

# Check games per team per season
games_summary = game_number_mapping.groupby(['season', 'team_name'])['game_number'].max().reset_index()
games_summary.columns = ['season', 'team', 'total_games']

print(f"\n\nGames per team per season summary:")
print(f"Teams with 38 games: {(games_summary['total_games'] == 38).sum()}")
print(f"Teams with < 38 games: {(games_summary['total_games'] < 38).sum()}")

# Show seasons breakdown
print("\nBy season:")
for season in sorted(games_summary['season'].unique()):
    season_data = games_summary[games_summary['season'] == season]
    print(f"  {season}: {len(season_data)} teams, games range: {season_data['total_games'].min()}-{season_data['total_games'].max()}")

VERIFICATION: Sample of game_number mapping

Liverpool 2019-20 first 10 games:
     team_name  game_number  gw              kickoff_time  fixture_id
342  Liverpool            1   1 2019-08-09 19:00:00+00:00           1
343  Liverpool            2   2 2019-08-17 14:00:00+00:00          19
344  Liverpool            3   3 2019-08-24 16:30:00+00:00          24
345  Liverpool            4   4 2019-08-31 16:30:00+00:00          32
346  Liverpool            5   5 2019-09-14 11:30:00+00:00          44
347  Liverpool            6   6 2019-09-22 15:30:00+00:00          53
348  Liverpool            7   7 2019-09-28 11:30:00+00:00          68
349  Liverpool            8   8 2019-10-05 14:00:00+00:00          74
350  Liverpool            9   9 2019-10-20 15:30:00+00:00          87
351  Liverpool           10  10 2019-10-27 16:30:00+00:00          94


Games per team per season summary:
Teams with 38 games: 120
Teams with < 38 games: 20

By season:
  2019-20: 20 teams, games range: 38-38
  2020-21: 

## Step 2: Join game_number to all_seasons_data.csv

Join using the fixture ID (100% match rate for seasons with fixtures.csv)

In [68]:
def add_game_number_to_all_seasons(df, mapping):
    """
    Add game_number to all_seasons_data using fixture ID matching.
    
    Args:
        df: all_seasons_data DataFrame
        mapping: game_number_mapping DataFrame from Step 1
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Create slim mapping for joining: (season, fixture_id, team_name) → game_number
    # We need team_name because each fixture has 2 teams
    slim_mapping = mapping[['season', 'fixture_id', 'team_name', 'game_number']].copy()
    slim_mapping = slim_mapping.rename(columns={'fixture_id': 'fixture', 'team_name': 'team'})
    
    print(f"Original records: {len(df):,}")
    print(f"Mapping records: {len(slim_mapping):,}")
    
    # Merge on season, fixture, and team
    df = df.merge(
        slim_mapping,
        on=['season', 'fixture', 'team'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to all_seasons_data
print("="*80)
print("STEP 2: Adding game_number to all_seasons_data.csv")
print("="*80)
all_seasons_updated = add_game_number_to_all_seasons(all_seasons_df, game_number_mapping)

STEP 2: Adding game_number to all_seasons_data.csv
Dropped existing game_number column
Original records: 216,537
Mapping records: 4,740

✅ Matched records: 148,479 (68.6%)
❌ Unmatched records: 68,058 (31.4%)

Unmatched by season:
season
2016-17    23679
2017-18    22467
2018-19    21790
2019-20      122
dtype: int64


In [69]:
# Verify all_seasons_data results
print("="*80)
print("VERIFICATION: all_seasons_data game_number results")
print("="*80)

# Check games per team per season
games_per_team = all_seasons_updated.groupby(['season', 'team'])['game_number'].max().reset_index()
games_per_team.columns = ['season', 'team', 'total_games']

print(f"\nGames per team per season:")
print(f"  - Total team-seasons: {len(games_per_team)}")
print(f"  - Teams with 38 games: {(games_per_team['total_games'] == 38).sum()}")
print(f"  - Min games: {games_per_team['total_games'].min()}")
print(f"  - Max games: {games_per_team['total_games'].max()}")

# Show by season
print("\nBy season:")
for season in sorted(games_per_team['season'].unique()):
    season_data = games_per_team[games_per_team['season'] == season]
    with_38 = (season_data['total_games'] == 38).sum()
    total = len(season_data)
    print(f"  {season}: {total} teams, {with_38}/{total} with 38 games, range: {season_data['total_games'].min()}-{season_data['total_games'].max()}")

# Sample check - Liverpool 2019-20
print("\n" + "="*80)
print("SAMPLE: Liverpool 2019-20 first 10 games")
print("="*80)
liverpool_sample = all_seasons_updated[
    (all_seasons_updated['season'] == '2019-20') & 
    (all_seasons_updated['team'] == 'Liverpool')
].groupby('game_number')[['kickoff_time', 'opponent_team', 'GW', 'was_home']].first().head(10)
print(liverpool_sample)

VERIFICATION: all_seasons_data game_number results

Games per team per season:
  - Total team-seasons: 200
  - Teams with 38 games: 100
  - Min games: 9
  - Max games: 38

By season:
  2016-17: 20 teams, 0/20 with 38 games, range: <NA>-<NA>
  2017-18: 20 teams, 0/20 with 38 games, range: <NA>-<NA>
  2018-19: 20 teams, 0/20 with 38 games, range: <NA>-<NA>
  2019-20: 20 teams, 20/20 with 38 games, range: 38-38
  2020-21: 20 teams, 20/20 with 38 games, range: 38-38
  2021-22: 20 teams, 20/20 with 38 games, range: 38-38
  2022-23: 20 teams, 20/20 with 38 games, range: 38-38
  2023-24: 20 teams, 20/20 with 38 games, range: 38-38
  2024-25: 20 teams, 0/20 with 38 games, range: 20-21
  2025-26: 20 teams, 0/20 with 38 games, range: 9-9

SAMPLE: Liverpool 2019-20 first 10 games
                          kickoff_time  opponent_team  GW  was_home
game_number                                                        
1            2019-08-09 19:00:00+00:00        Norwich   1      True
2            201

## Step 3: Add game_number to defensive_stats_raw.csv

Since defensive_stats doesn't have fixture IDs, we'll use date-based matching:
1. Parse the 'game' column to extract date and teams
2. Match with fixtures using date + team combination

In [73]:
def add_game_number_to_defensive_stats(df, mapping):
    """
    Add game_number to defensive_stats using date-based matching.
    
    The 'game' column format: "YYYY-MM-DD Team1-Team2"
    The 'season' column format: "1920" (for 2019-20)
    
    Args:
        df: defensive_stats DataFrame
        mapping: game_number_mapping DataFrame
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Team name mapping: defensive_stats name → mapping name (FPL short name)
    team_name_mapping = {
        'Brighton & Hove Albion': 'Brighton',
        'Ipswich Town': 'Ipswich',
        'Leeds United': 'Leeds',
        'Leicester City': 'Leicester',
        'Luton Town': 'Luton',
        'Manchester City': 'Man City',
        'Manchester United': 'Man Utd',
        'Newcastle United': 'Newcastle',
        'Norwich City': 'Norwich',
        'Nottingham Forest': "Nott'm Forest",
        'Sheffield United': 'Sheffield Utd',
        'Tottenham Hotspur': 'Spurs',
        'West Bromwich Albion': 'West Brom',
        'West Ham United': 'West Ham',
        'Wolverhampton Wanderers': 'Wolves',
    }
    
    # Map team names to match FPL naming
    df['team_mapped'] = df['team'].map(team_name_mapping).fillna(df['team'])
    
    # Extract date from 'game' column (format: "YYYY-MM-DD Team1-Team2")
    df['game_date'] = pd.to_datetime(df['game'].str.extract(r'^(\d{4}-\d{2}-\d{2})')[0], errors='coerce')
    
    # Convert defensive stats season format (1920) to standard format (2019-20)
    def convert_season(s):
        if pd.isna(s):
            return None
        s = str(s).replace('.0', '')
        if len(s) == 4:  # e.g., "1920"
            return f"20{s[:2]}-{s[2:]}"
        return s
    
    df['season_standard'] = df['season'].apply(convert_season)
    
    # Create date-based mapping from fixtures
    mapping_for_date = mapping.copy()
    mapping_for_date['game_date'] = mapping_for_date['kickoff_time'].dt.date
    mapping_for_date['game_date'] = pd.to_datetime(mapping_for_date['game_date'])
    
    # Create slim mapping: (season, team_name, game_date) → game_number
    date_mapping = mapping_for_date[['season', 'team_name', 'game_date', 'game_number']].copy()
    date_mapping = date_mapping.rename(columns={'team_name': 'team_mapped', 'season': 'season_standard'})
    
    # Remove duplicates (same team can't have 2 games on same day)
    date_mapping = date_mapping.drop_duplicates(subset=['season_standard', 'team_mapped', 'game_date'])
    
    print(f"Original records: {len(df):,}")
    print(f"Date mapping records: {len(date_mapping):,}")
    
    # Merge
    df = df.merge(
        date_mapping,
        on=['season_standard', 'team_mapped', 'game_date'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Clean up temporary columns
    df = df.drop(['game_date', 'season_standard', 'team_mapped'], axis=1)
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to defensive_stats
print("="*80)
print("STEP 3: Adding game_number to defensive_stats_raw.csv")
print("="*80)
defensive_stats_updated = add_game_number_to_defensive_stats(defensive_stats_df, game_number_mapping)

STEP 3: Adding game_number to defensive_stats_raw.csv
Dropped existing game_number column
Original records: 65,789
Date mapping records: 4,740

✅ Matched records: 65,788 (100.0%)
❌ Unmatched records: 1 (0.0%)

Unmatched by season:
Series([], dtype: int64)


In [74]:
# Verify defensive_stats results
print("="*80)
print("VERIFICATION: defensive_stats game_number results")
print("="*80)

# Check games per team per season
def_games_per_team = defensive_stats_updated.groupby(['season', 'team'])['game_number'].max().reset_index()
def_games_per_team.columns = ['season', 'team', 'total_games']

print(f"\nGames per team per season:")
print(f"  - Total team-seasons: {len(def_games_per_team)}")
print(f"  - Teams with 38 games: {(def_games_per_team['total_games'] == 38).sum()}")
print(f"  - Min games: {def_games_per_team['total_games'].min()}")
print(f"  - Max games: {def_games_per_team['total_games'].max()}")

# Show by season
print("\nBy season:")
for season in sorted(def_games_per_team['season'].dropna().unique()):
    season_data = def_games_per_team[def_games_per_team['season'] == season]
    with_38 = (season_data['total_games'] == 38).sum()
    total = len(season_data)
    print(f"  {season}: {total} teams, {with_38}/{total} with 38 games, range: {season_data['total_games'].min()}-{season_data['total_games'].max()}")

VERIFICATION: defensive_stats game_number results

Games per team per season:
  - Total team-seasons: 120
  - Teams with 38 games: 120
  - Min games: 38
  - Max games: 38

By season:
  1920.0: 20 teams, 20/20 with 38 games, range: 38-38
  2021.0: 20 teams, 20/20 with 38 games, range: 38-38
  2122.0: 20 teams, 20/20 with 38 games, range: 38-38
  2223.0: 20 teams, 20/20 with 38 games, range: 38-38
  2324.0: 20 teams, 20/20 with 38 games, range: 38-38
  2425.0: 20 teams, 20/20 with 38 games, range: 38-38


## Step 4: Comprehensive Validation

In [75]:
# VALIDATION 1: Check game_number sequences are continuous (no gaps)
print("="*80)
print("VALIDATION 1: Check for gaps in game_number sequences")
print("="*80)

gaps_found = []
for (season, team), group in all_seasons_updated.groupby(['season', 'team']):
    unique_gn = sorted(group['game_number'].dropna().unique())
    if len(unique_gn) > 0:
        expected = list(range(1, len(unique_gn) + 1))
        if unique_gn != expected:
            gaps_found.append({
                'season': season,
                'team': team,
                'actual': unique_gn[:10],  # Show first 10
                'expected': expected[:10]
            })

if len(gaps_found) == 0:
    print("✅ All team-seasons have continuous game_number sequences!")
else:
    print(f"⚠️ Found {len(gaps_found)} team-seasons with gaps:")
    for item in gaps_found[:5]:
        print(f"  - {item['season']} {item['team']}")

VALIDATION 1: Check for gaps in game_number sequences
⚠️ Found 20 team-seasons with gaps:
  - 2025-26 Arsenal
  - 2025-26 Aston Villa
  - 2025-26 Bournemouth
  - 2025-26 Brentford
  - 2025-26 Brighton


In [76]:
# VALIDATION 2: Check chronological order
print("="*80)
print("VALIDATION 2: Verify games are in chronological order")
print("="*80)

order_issues = []
for (season, team), group in all_seasons_updated.groupby(['season', 'team']):
    # Get one record per game_number with kickoff_time
    game_times = group.groupby('game_number')['kickoff_time'].first().sort_index()
    times_list = pd.to_datetime(game_times).tolist()
    
    for i in range(len(times_list) - 1):
        if pd.notna(times_list[i]) and pd.notna(times_list[i+1]):
            if times_list[i] > times_list[i+1]:
                order_issues.append({
                    'season': season,
                    'team': team,
                    'game': i+1
                })
                break

if len(order_issues) == 0:
    print("✅ All games are in correct chronological order!")
else:
    print(f"⚠️ Found {len(order_issues)} team-seasons with order issues:")
    for item in order_issues[:5]:
        print(f"  - {item['season']} {item['team']} at game {item['game']}")

VALIDATION 2: Verify games are in chronological order
✅ All games are in correct chronological order!


In [77]:
# VALIDATION 3: Compare GW vs game_number (expected to differ due to postponements)
print("="*80)
print("VALIDATION 3: GW vs game_number comparison")
print("="*80)

gw_game_comp = all_seasons_updated.groupby(['season', 'team', 'game_number'])['GW'].first().reset_index()
gw_game_comp['difference'] = gw_game_comp['GW'] - gw_game_comp['game_number']

mismatches = gw_game_comp[gw_game_comp['difference'] != 0]

print(f"Total games: {len(gw_game_comp):,}")
print(f"GW matches game_number: {len(gw_game_comp) - len(mismatches):,} ({(len(gw_game_comp) - len(mismatches))/len(gw_game_comp)*100:.1f}%)")
print(f"GW differs from game_number: {len(mismatches):,} ({len(mismatches)/len(gw_game_comp)*100:.1f}%)")
print("\nNote: Differences are EXPECTED due to postponed/rescheduled matches.")
print("game_number = true chronological order (when game was actually played)")
print("GW = scheduled gameweek (may not reflect actual play order)")

VALIDATION 3: GW vs game_number comparison
Total games: 4,378
GW matches game_number: 3,080 (70.4%)
GW differs from game_number: 1,298 (29.6%)

Note: Differences are EXPECTED due to postponed/rescheduled matches.
game_number = true chronological order (when game was actually played)
GW = scheduled gameweek (may not reflect actual play order)


In [78]:
# FINAL SUMMARY
print("="*80)
print("FINAL SUMMARY - Data Quality Report")
print("="*80)

print("\n📊 ALL_SEASONS_DATA.CSV:")
print(f"  • Total records: {len(all_seasons_updated):,}")
print(f"  • Records with game_number: {all_seasons_updated['game_number'].notna().sum():,} ({all_seasons_updated['game_number'].notna().sum()/len(all_seasons_updated)*100:.1f}%)")
print(f"  • Game number range: {all_seasons_updated['game_number'].min()} to {all_seasons_updated['game_number'].max()}")
print(f"  • Total seasons: {all_seasons_updated['season'].nunique()}")
print(f"  • Total teams: {all_seasons_updated['team'].nunique()}")

# Count teams with 38 games
all_games = all_seasons_updated.groupby(['season', 'team'])['game_number'].max().reset_index()
teams_38 = (all_games['game_number'] == 38).sum()
print(f"  • Team-seasons with 38 games: {teams_38}/{len(all_games)} ({teams_38/len(all_games)*100:.1f}%)")

print("\n📊 DEFENSIVE_STATS_RAW.CSV:")
print(f"  • Total records: {len(defensive_stats_updated):,}")
print(f"  • Records with game_number: {defensive_stats_updated['game_number'].notna().sum():,} ({defensive_stats_updated['game_number'].notna().sum()/len(defensive_stats_updated)*100:.1f}%)")
print(f"  • Game number range: {defensive_stats_updated['game_number'].min()} to {defensive_stats_updated['game_number'].max()}")

# Count teams with 38 games
def_games = defensive_stats_updated.groupby(['season', 'team'])['game_number'].max().reset_index()
def_teams_38 = (def_games['game_number'] == 38).sum()
print(f"  • Team-seasons with 38 games: {def_teams_38}/{len(def_games)} ({def_teams_38/len(def_games)*100:.1f}%)")

print("\n" + "="*80)
print("✅ GAME_NUMBER ASSIGNMENT COMPLETE!")
print("="*80)

FINAL SUMMARY - Data Quality Report

📊 ALL_SEASONS_DATA.CSV:
  • Total records: 216,537
  • Records with game_number: 148,479 (68.6%)
  • Game number range: 1 to 38
  • Total seasons: 10
  • Total teams: 34
  • Team-seasons with 38 games: 100/200 (50.0%)

📊 DEFENSIVE_STATS_RAW.CSV:
  • Total records: 65,789
  • Records with game_number: 65,788 (100.0%)
  • Game number range: 1 to 38
  • Team-seasons with 38 games: 120/120 (100.0%)

✅ GAME_NUMBER ASSIGNMENT COMPLETE!


## Step 5: Save Updated Datasets

In [79]:
# Save the updated datasets
print("Saving updated datasets...")
print("="*80)

# Save all_seasons_data with game_number
all_seasons_updated.to_csv('all_seasons_data.csv', index=False)
print(f"✓ Saved: all_seasons_data.csv")
print(f"  - {len(all_seasons_updated):,} records")
print(f"  - game_number range: 1-{all_seasons_updated['game_number'].max()}")

# Save defensive_stats with game_number
defensive_stats_updated.to_csv('defensive_stats_raw.csv', index=False)
print(f"\n✓ Saved: defensive_stats_raw.csv")
print(f"  - {len(defensive_stats_updated):,} records")
print(f"  - game_number range: 1-{defensive_stats_updated['game_number'].max()}")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)

Saving updated datasets...
✓ Saved: all_seasons_data.csv
  - 216,537 records
  - game_number range: 1-38

✓ Saved: defensive_stats_raw.csv
  - 65,789 records
  - game_number range: 1-38

ALL DONE!
